In [ ]:
# ── Colab Setup ──────────────────────────────────────────────────────────────
# Run this cell first. It mounts Google Drive and adds the project's src/
# directory to Python's path so grammar_loader can be imported.
# If running locally, it just makes sure src/ is on the path.

import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # ← Update this to match where MScProject lives in your Google Drive
    PROJECT_ROOT = '/content/drive/MyDrive/MScProject'

    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
    os.chdir(PROJECT_ROOT)
    print(f"Working directory: {os.getcwd()}")
else:
    # Running locally — add src/ to path if needed
    src_dir = os.path.join(os.getcwd(), 'src')
    if os.path.isdir(src_dir) and src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    print("Running locally.")

# 02 — Train Language Model

Trains an **LSTM** or **Transformer** language model on grammar strings using next-token prediction.

The trained model and vocabulary are saved as a checkpoint for use in `03_probe.ipynb`.

Key design choices (from Barry's notes):
- CLS and SEP tokens wrap each sequence; PAD tokens are ignored in the loss
- Transformer can be run in **causal** (GPT-style) or **bidirectional** (BERT-style) mode
- LSTM uses `LSTMCell` layer-by-layer so hidden states are accessible for probing

In [ ]:
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

from grammar_loader import load_grammar, build_vocab, tokenize

## Configuration

In [ ]:
DATA_FILE       = 'data/anbn_n1-100.txt'   # output of 01_generate_data.ipynb
GRAMMAR_FILE    = 'grammars/anbn.txt'       # grammar file (for vocabulary)

# ── Model selection ───────────────────────────────────────────────────────────
# 'lstm'        → custom LSTM trained from scratch
# 'transformer' → custom Transformer trained from scratch
# 'bert'        → bert-base-uncased fine-tuned on grammar strings (MLM)
# 'gpt2'        → gpt2 fine-tuned on grammar strings (causal LM)
MODEL_TYPE      = 'lstm'
CAUSAL_MASK     = True    # Transformer only: True=causal, False=bidirectional

TRAIN_MAX_N     = 50    # train on n=1..50; test generalisation on n=51..100

# Hyperparameters (custom models)
EPOCHS          = 50
BATCH_SIZE      = 32
LR              = 1e-3
EMBED_DIM       = 64
HIDDEN_DIM      = 256
NUM_LAYERS      = 2

# Hyperparameters (HuggingFace fine-tuning)
HF_EPOCHS       = 10
HF_BATCH_SIZE   = 16
HF_LR           = 2e-5    # standard fine-tuning LR for BERT/GPT-2
MLM_PROBABILITY = 0.15    # fraction of tokens masked for BERT MLM

CHECKPOINT_DIR  = 'checkpoints'

## Dataset

In [ ]:
class GrammarDataset(Dataset):
    """Loads grammar strings and returns token-ID sequences."""

    def __init__(self, filepath: str, vocab: dict):
        self.vocab = vocab
        self.sequences = []
        with open(filepath) as f:
            for line in f:
                line = line.strip()
                if line:
                    ids = tokenize(line, vocab, add_special=True)
                    self.sequences.append(torch.tensor(ids, dtype=torch.long))

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx]


def collate_fn(batch, pad_id: int):
    """Pad sequences in a batch to the same length."""
    max_len = max(seq.size(0) for seq in batch)
    padded = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    for i, seq in enumerate(batch):
        padded[i, :seq.size(0)] = seq
    return padded

## Models

In [ ]:
class LSTMLanguageModel(nn.Module):
    """
    LSTM language model built from a stack of LSTMCells.

    Using LSTMCell instead of nn.LSTM gives direct access to each layer's
    hidden state at every timestep, which is required for layer-wise probing.
    """

    def __init__(self, vocab_size: int, embed_dim: int = 64, hidden_dim: int = 256,
                 num_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.cells = nn.ModuleList([
            nn.LSTMCell(embed_dim if i == 0 else hidden_dim, hidden_dim)
            for i in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.output_proj = nn.Linear(hidden_dim, vocab_size)
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

    def forward(self, x, return_hidden=False):
        """
        x: (batch, seq_len) token IDs
        Returns logits (batch, seq_len, vocab_size).
        If return_hidden=True, also returns a list of (B, T, H) tensors — one per layer.
        """
        B, T = x.shape
        emb = self.embedding(x)                         # (B, T, E)

        h = [torch.zeros(B, self.hidden_dim, device=x.device) for _ in self.cells]
        c = [torch.zeros(B, self.hidden_dim, device=x.device) for _ in self.cells]
        layer_outputs = [[] for _ in self.cells]

        for t in range(T):
            inp = emb[:, t, :]                          # (B, E)
            for i, cell in enumerate(self.cells):
                h[i], c[i] = cell(inp, (h[i], c[i]))   # (B, H)
                inp = self.dropout(h[i])
                layer_outputs[i].append(h[i])

        layer_hiddens = [torch.stack(steps, dim=1) for steps in layer_outputs]
        logits = self.output_proj(layer_hiddens[-1])    # (B, T, V)

        if return_hidden:
            return logits, layer_hiddens
        return logits


class TransformerLanguageModel(nn.Module):
    """
    Transformer language model.

    Use causal_mask=True  for autoregressive (GPT-style) training.
    Use causal_mask=False for bidirectional (BERT-style) training.
    Barry's note: test both — bidirectional should handle closing brackets better.
    """

    def __init__(self, vocab_size: int, embed_dim: int = 64, num_heads: int = 4,
                 num_layers: int = 2, ff_dim: int = 256, dropout: float = 0.1,
                 max_seq_len: int = 512, causal_mask: bool = True):
        super().__init__()
        self.causal_mask = causal_mask
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_seq_len, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_proj = nn.Linear(embed_dim, vocab_size)
        self.num_layers = num_layers

    def forward(self, x, return_hidden=False):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0)
        emb = self.embedding(x) + self.pos_embedding(positions)

        pad_mask = (x == 0)
        causal = None
        if self.causal_mask:
            causal = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)

        out = self.transformer(emb, mask=causal, src_key_padding_mask=pad_mask)
        logits = self.output_proj(out)

        if return_hidden:
            return logits, out
        return logits

## Build vocabulary and dataset

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

grammar = load_grammar(GRAMMAR_FILE)
vocab = build_vocab(grammar)
pad_id = vocab['[PAD]']
print(f"Vocabulary ({len(vocab)} tokens): {vocab}")

dataset = GrammarDataset(DATA_FILE, vocab)

# Split by n — sequences are stored in order n=1,2,...,MAX_N
# so the first TRAIN_MAX_N entries correspond to n=1..TRAIN_MAX_N
train_seqs = dataset.sequences[:TRAIN_MAX_N]
test_seqs  = dataset.sequences[TRAIN_MAX_N:]

train_loader = DataLoader(
    train_seqs, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=lambda b: collate_fn(b, pad_id),
)
test_loader = DataLoader(
    test_seqs, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=lambda b: collate_fn(b, pad_id),
)

print(f"Train: {len(train_seqs)} sequences  (n=1..{TRAIN_MAX_N})")
print(f"Test:  {len(test_seqs)} sequences  (n={TRAIN_MAX_N + 1}..{len(dataset)})")

## Build model

In [ ]:
if MODEL_TYPE in ('bert', 'gpt2'):
    print(f"Skipping custom model build — MODEL_TYPE is '{MODEL_TYPE}'. Using HuggingFace models below.")
else:
    vocab_size = len(vocab)

    if MODEL_TYPE == 'lstm':
        model = LSTMLanguageModel(
            vocab_size=vocab_size,
            embed_dim=EMBED_DIM,
            hidden_dim=HIDDEN_DIM,
            num_layers=NUM_LAYERS,
        )
    else:
        model = TransformerLanguageModel(
            vocab_size=vocab_size,
            embed_dim=EMBED_DIM,
            num_heads=4,
            num_layers=NUM_LAYERS,
            ff_dim=HIDDEN_DIM,
            causal_mask=CAUSAL_MASK,
        )

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    print(f"Model: {MODEL_TYPE}  params: {sum(p.numel() for p in model.parameters()):,}")


## Training loop

In [ ]:
if MODEL_TYPE in ('bert', 'gpt2'):
    print(f"Skipping custom training loop — MODEL_TYPE is '{MODEL_TYPE}'.")
else:
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            batch = batch.to(device)
            inputs  = batch[:, :-1]
            targets = batch[:, 1:]
    
            logits = model(inputs)
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
    
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
    
        avg_loss = total_loss / len(train_loader)
        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:>4}/{EPOCHS}  loss: {avg_loss:.4f}")

## Save checkpoint

In [ ]:
if MODEL_TYPE in ('bert', 'gpt2'):
    print(f"Skipping custom checkpoint save — MODEL_TYPE is '{MODEL_TYPE}'.")
else:
    Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
    
    if MODEL_TYPE == 'lstm':
        tag = 'lstm'
    else:
        tag = f'transformer_{"causal" if CAUSAL_MASK else "no_causal"}'
    
    checkpoint_path = f"{CHECKPOINT_DIR}/{grammar.name}_{tag}.pt"
    
    torch.save({
        'model_state': model.state_dict(),
        'vocab': vocab,
        'grammar_name': grammar.name,
        'model_type': MODEL_TYPE,
        'causal_mask': CAUSAL_MASK,
        'args': {
            'embed_dim': EMBED_DIM,
            'hidden_dim': HIDDEN_DIM,
            'num_layers': NUM_LAYERS,
        },
    }, checkpoint_path)
    
    print(f"Saved: {checkpoint_path}")

---

## HuggingFace Fine-tuning — BERT and GPT-2

Run the cells below when `MODEL_TYPE` is `'bert'` or `'gpt2'`.

**BERT** (`bert-base-uncased`) is fine-tuned with **Masked Language Modelling** (MLM):
randomly mask 15% of tokens, train the model to predict them.  This is the
natural pre-training objective for BERT and tests whether the model can infer
missing symbols from grammatical context.

**GPT-2** is fine-tuned with **causal language modelling** (next-token prediction),
identical to the custom Transformer above but using GPT-2's pre-trained weights
as the starting point.

Checkpoints are saved as `checkpoints/<grammar>_bert.pt` /
`checkpoints/<grammar>_gpt2.pt` alongside the custom-model checkpoints so that
`03_probe.ipynb` and `04_attention.ipynb` can load them uniformly.

In [ ]:
if MODEL_TYPE not in ('bert', 'gpt2'):
    print(f"Skipping HuggingFace section — MODEL_TYPE is '{MODEL_TYPE}'. Only run for 'bert' or 'gpt2'.")
else:
    from transformers import (
        AutoTokenizer, AutoModelForMaskedLM, AutoModelForCausalLM,
        DataCollatorForLanguageModeling, DataCollatorWithPadding,
        get_linear_schedule_with_warmup,
    )
    from torch.utils.data import Dataset as TorchDataset, DataLoader as TorchDataLoader

    # ── Load HuggingFace tokenizer and model ────────────────────────────────────
    HF_MODEL_NAME = 'bert-base-uncased' if MODEL_TYPE == 'bert' else 'gpt2'

    hf_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME)

    # GPT-2 has no pad token by default — use EOS as pad
    if MODEL_TYPE == 'gpt2':
        hf_tokenizer.pad_token = hf_tokenizer.eos_token

    if MODEL_TYPE == 'bert':
        hf_model = AutoModelForMaskedLM.from_pretrained(HF_MODEL_NAME)
    else:
        hf_model = AutoModelForCausalLM.from_pretrained(HF_MODEL_NAME)

    hf_model = hf_model.to(device)
    print(f"Loaded {HF_MODEL_NAME}  ({sum(p.numel() for p in hf_model.parameters()):,} params)")

In [ ]:
if MODEL_TYPE in ('bert', 'gpt2'):
    # ── Dataset: read raw strings and tokenise with HF tokenizer ────────────────

    class HFGrammarDataset(TorchDataset):
        """
        Loads grammar strings and tokenises them using the HuggingFace tokenizer.

        Each string is a grammatical sequence like 'aaabbb' or 'aaabbbccc'.
        We tokenise at the character level (each symbol maps to one token in both
        BERT and GPT-2 vocabularies).

        max_length is computed from the data so long sequences (e.g. aⁿbⁿcⁿ at
        large n) are never silently truncated.
        """
        def __init__(self, filepath: str, tokenizer):
            with open(filepath) as f:
                strings = [l.strip() for l in f if l.strip()]

            # Insert spaces between characters so the tokenizer treats each as a
            # separate token (e.g. 'aabb' → 'a a b b'). This guarantees 1:1
            # character-to-token alignment needed for probing.
            spaced = [' '.join(list(s)) for s in strings]

            # Compute max_length from actual data (+2 for CLS/SEP) so sequences
            # are never truncated, regardless of grammar or MAX_N.
            max_length = max(len(s.split()) for s in spaced) + 2

            print(f"HF tokeniser: longest sequence = {max_length - 2} tokens  "
                  f"(max_length set to {max_length})")

            enc = tokenizer(
                spaced,
                padding        = 'max_length',
                truncation     = True,
                max_length     = max_length,
                return_tensors = 'pt',
            )
            self.input_ids      = enc['input_ids']
            self.attention_mask = enc['attention_mask']

        def __len__(self):
            return len(self.input_ids)

        def __getitem__(self, idx):
            return {
                'input_ids':      self.input_ids[idx],
                'attention_mask': self.attention_mask[idx],
            }


    hf_dataset = HFGrammarDataset(DATA_FILE, hf_tokenizer)

    hf_train = TorchDataLoader(
        [hf_dataset[i] for i in range(min(TRAIN_MAX_N, len(hf_dataset)))],
        batch_size=HF_BATCH_SIZE, shuffle=True,
    )
    hf_test = TorchDataLoader(
        [hf_dataset[i] for i in range(TRAIN_MAX_N, len(hf_dataset))],
        batch_size=HF_BATCH_SIZE, shuffle=False,
    )
    print(f"HF dataset: {len(hf_dataset)} sequences  "
          f"(train: {TRAIN_MAX_N}, test: {len(hf_dataset) - TRAIN_MAX_N})")

In [ ]:
if MODEL_TYPE in ('bert', 'gpt2'):
    import random

    def mask_tokens_for_bert(input_ids, attention_mask, tokenizer, mlm_prob=0.15):
        """
        Apply random token masking for BERT MLM.
        Returns (masked_input_ids, labels) where labels=-100 at non-masked positions.
        """
        labels = input_ids.clone()
        prob_matrix = torch.full(labels.shape, mlm_prob)

        # Don't mask special tokens ([CLS], [SEP], [PAD])
        special_ids = set(tokenizer.all_special_ids)
        for sid in special_ids:
            prob_matrix[input_ids == sid] = 0.0
        # Don't mask padding
        prob_matrix[attention_mask == 0] = 0.0

        masked = torch.bernoulli(prob_matrix).bool()
        labels[~masked] = -100   # only compute loss on masked positions

        input_ids[masked] = tokenizer.mask_token_id
        return input_ids, labels


    # ── Fine-tuning loop ─────────────────────────────────────────────────────────
    optimizer = torch.optim.AdamW(hf_model.parameters(), lr=HF_LR)
    total_steps = len(hf_train) * HF_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps   = total_steps // 10,
        num_training_steps = total_steps,
    )

    for epoch in range(1, HF_EPOCHS + 1):
        hf_model.train()
        total_loss = 0.0

        for batch in hf_train:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            if MODEL_TYPE == 'bert':
                # MLM: randomly mask tokens and predict them
                input_ids, labels = mask_tokens_for_bert(
                    input_ids.clone(), attention_mask.clone(), hf_tokenizer, MLM_PROBABILITY
                )
                labels = labels.to(device)
                outputs = hf_model(
                    input_ids      = input_ids,
                    attention_mask = attention_mask,
                    labels         = labels,
                )
            else:
                # GPT-2: causal LM — labels = input_ids (HuggingFace shifts internally)
                outputs = hf_model(
                    input_ids      = input_ids,
                    attention_mask = attention_mask,
                    labels         = input_ids,
                )

            loss = outputs.loss
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(hf_model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(hf_train)
        if epoch % 2 == 0 or epoch == 1:
            print(f"Epoch {epoch:>3}/{HF_EPOCHS}  loss: {avg_loss:.4f}")

In [ ]:
if MODEL_TYPE in ('bert', 'gpt2'):
    # ── Save HuggingFace checkpoint ──────────────────────────────────────────────
    from grammar_loader import load_grammar as _load_grammar
    _grammar = _load_grammar(GRAMMAR_FILE)

    ckpt_dir = f"{CHECKPOINT_DIR}/{_grammar.name}_{MODEL_TYPE}"
    hf_model.save_pretrained(ckpt_dir)
    hf_tokenizer.save_pretrained(ckpt_dir)

    import json
    meta = {
        'grammar_name': _grammar.name,
        'model_type':   MODEL_TYPE,
        'hf_model':     HF_MODEL_NAME,
        'train_max_n':  TRAIN_MAX_N,
    }
    with open(f"{ckpt_dir}/meta.json", 'w') as f:
        json.dump(meta, f, indent=2)

    print(f"Saved to: {ckpt_dir}/")

## Generalisation evaluation

Tests the trained model on held-out sequences (n=51..100) that it never saw during training.
This is the core question: did the model learn the *rule* (aⁿbⁿ) or just memorise the examples?

**Next-token accuracy** at each position tells us whether the model predicts the correct symbol.
Pay attention to the positions where the grammar *switches* from `a` to `b` — that's where
a model that has truly learned counting will succeed and a memoriser will fail.

In [ ]:
if MODEL_TYPE in ('bert', 'gpt2'):
    print(f"Skipping generalisation eval — MODEL_TYPE is '{MODEL_TYPE}'. "
          "Custom model evaluation only applies to 'lstm' / 'transformer'.")
else:
    model.eval()
    total_correct = 0
    total_tokens  = 0
    
    with torch.no_grad():
        for batch in test_loader:
            batch   = batch.to(device)
            inputs  = batch[:, :-1]
            targets = batch[:, 1:]
    
            logits  = model(inputs)                              # (B, T-1, V)
            preds   = logits.argmax(dim=-1)                      # (B, T-1)
    
            # Ignore PAD positions
            mask = (targets != pad_id)
            total_correct += (preds[mask] == targets[mask]).sum().item()
            total_tokens  += mask.sum().item()
    
    gen_acc = total_correct / total_tokens if total_tokens > 0 else 0.0
    print(f"Generalisation accuracy  (n={TRAIN_MAX_N + 1}..{len(dataset)}): {gen_acc:.4f}  "
          f"({total_correct}/{total_tokens} tokens correct)")